<a href="https://colab.research.google.com/github/luisaespinoza/CSCI164-Problem-Solving-w-Search/blob/main/Problem-Solving-w-Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import heapq

# Tile Sliding Domain: Initial State Space

In [50]:
# StateDimension=3
# InitialState = [1,2,3,4,5,6,0,7,8]
# GoalState=[1,2,3,4,5,6,7,8,0]
# StateDimension = 4  # Update to 4x4
# InitialState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 13, 14, 15]
# GoalState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
def configure_puzzle(dimension,initial_state=None):
    if dimension == 3:
        if initial_state is None:
          InitialState = [1, 2, 3, 4, 5, 6, 7, 8, 0]
        else:
          InitialState=initial_state
        GoalState = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    elif dimension == 4:
        if initial_state is None:
          InitialState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
        else:
          InitialState=initial_state
        GoalState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
    else:
        raise ValueError("Invalid puzzle dimension. Choose 3 or 4.")

    global StateDimension, Goal
    StateDimension = dimension
    Goal = tuple(GoalState)

    return InitialState,Goal
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

In [4]:
def Result(state, action):
  i = state.index(0)
  newState = list(state)
  row,col=i//StateDimension, i % StateDimension
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return newState
  if action=='u':
    l,r = row*StateDimension+col, (row-1)*StateDimension+col
  elif action=='d':
    l,r = row*StateDimension+col, (row+1)*StateDimension+col
  elif action=='l':
    l,r = row*StateDimension+col, row*StateDimension+col-1
  elif action=='r' :
    l,r = row*StateDimension+col, row*StateDimension+col+1
  newState[l], newState[r] = newState[r], newState[l]
  return newState

def PrintState(s):
  for i in range(0,len(s),StateDimension):
    print(s[i:i+StateDimension])

# def LegalMove(state, action):
#   i = state.index(0)
#   row,col=i//StateDimension, i % StateDimension
#   newState = state.copy()
#   if ( (action=='u' and row==0) or
#        (action=='d' and row==StateDimension-1) or
#        (action=='l' and col==0) or
#        (action=='r' and col==StateDimension-1)):
#       return False
#   return True
def LegalMove(state, action, dimension):  # Added dimension argument
  i = state.index(0)
  row, col = i // dimension, i % dimension  # Use dimension instead of StateDimension
  if ( (action == 'u' and row == 0) or
       (action == 'd' and row == dimension - 1) or
       (action == 'l' and col == 0) or
       (action == 'r' and col == dimension - 1)):
      return False
  return True

In [5]:
def SingleTileManhattanDistance(tile, left, right):
  leftIndex = left.index(tile)
  rightIndex = right.index(tile)
  return (abs(leftIndex//StateDimension-rightIndex//StateDimension) +
          abs(leftIndex%StateDimension-rightIndex%StateDimension))

def ManhattanDistance(left, right):
  distances = [SingleTileManhattanDistance(tile, left, right)
     for tile in range(1, StateDimension**2)]
  ### print ("Distances= ", distances)
  return sum(distances)


In [6]:
def OutOfPlace(left, right):
  distances = [left[i]!=right[i] and right[i] != 0
     for i in range(StateDimension**2)]
  return sum(distances)

# **No longer neccesary with new modular approach**

In [6]:
PrintState(InitialState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]


In [7]:
PrintState(GoalState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [8]:
print("ManhattanDistance=  ", ManhattanDistance(InitialState, GoalState))
print("OutOfPlace= ", OutOfPlace(InitialState, GoalState))


ManhattanDistance=   3
OutOfPlace=  3


In [9]:
PrintState(InitialState)
print()
state1 = Result(InitialState, 'u')
PrintState(state1)
print()
state1 = Result(state1, 'r')
PrintState(state1)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]

[1, 2, 3, 4]
[5, 6, 7, 8]
[0, 10, 11, 12]
[9, 13, 14, 15]

[1, 2, 3, 4]
[5, 6, 7, 8]
[10, 0, 11, 12]
[9, 13, 14, 15]


# Random Walk

Take some random moves from a state and return the new state and the sequence of moves.

Do not include moves undoing last move, or having no effect.

In [7]:
# def RandomWalk(state, steps):
#   actionSequence = []
#   actionLast = None
#   for i in range(steps):
#     action = None
#     while action==None:
#       action = random.choice(Actions(state))
#       action = action if (LegalMove(state, action)
#           and action!= Opposite[actionLast]) else None
#     actionLast = action
#     state = Result(state, action)
#     actionSequence.append(action)
#   return state, actionSequence
def RandomWalk(state, steps, dimension):
    actionSequence = []
    actionLast = None

    for _ in range(steps):
        action = None
        while action is None:
            action = random.choice(Actions(state))
            # Ensure the action is legal and doesn't undo the last move
            if LegalMove(state, action, dimension) and action != Opposite[actionLast]:
                break
            else:
                action = None

        actionLast = action
        state = Result(state, action)  # Use Result with dimension
        actionSequence.append(action)

    return state, actionSequence

In [8]:
dimension=3
InitialState, GoalState = configure_puzzle(dimension)
state1, sol = RandomWalk(InitialState, 150,dimension)
PrintState(state1)
print (ManhattanDistance(state1, list(GoalState)), sol)

state1, sol = RandomWalk(InitialState, 5,dimension)
PrintState(InitialState)
print (sol)
PrintState(state1)

[7, 8, 0]
[5, 3, 1]
[4, 6, 2]
16 ['u', 'l', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'l', 'u', 'r', 'r', 'd', 'l', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'u', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'u', 'l', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'r', 'u', 'u', 'l', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'r', 'd', 'l', 'u', 'l', 'u', 'r', 'd', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'r', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'u', 'u', 'l', 'l', 'd', 'r', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'd', 'r', 'u', 'u']
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
['l', 'u', 'u', 'r', 'd']
[1, 3, 6]
[4, 2, 0]
[7, 5, 8]


In [9]:
def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

In [10]:
PrintState(InitialState)
print(['r','r'])
PrintState(ApplyMoves(['r','r'],InitialState))

[1, 2, 3]
[4, 5, 6]
[7, 8, 0]
['r', 'r']
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


In [11]:
def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

In [13]:
state1, sol = RandomWalk(GoalState, 5,dimension)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))


[1, 3, 6]
[4, 2, 0]
[7, 5, 8]
['l', 'u', 'u', 'r', 'd']
['u', 'l', 'd', 'd', 'r']
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


## Problem Class

INITIAL = InitialState  
IsGoal = Goal Test  
Actions = Actions List  
Result = Action Behavior  
ActionCost = Action Cost  

In [14]:
class Problem(object): pass

## Node

In [15]:
class Node(object):
  def __init__(self, state, parent=None, action=None, path_cost=0 ):
    self.State=state
    self.Parent=parent
    self.Action=action
    self.PathCost = path_cost

  def __str__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __repr__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __lt__(self, other):
    return self.PathCost < other.PathCost;

## Expand

In [40]:
def Expand(problem, node):
  ret = []
  s = node.State
  for action in problem.Actions(s):
    sPrime = problem.Result(s, action)
    cost =node.PathCost + problem.ActionCost(s,action,sPrime)
    ret.append(Node(sPrime, node, action, cost))
  return ret
def Solution(node):
    if node.Parent is None:
        return []
    return Solution(node.Parent) + [node.Action]

## Breadth-First Search

In [32]:
def BreadthFirstSearch(problem,f):
  node = Node(tuple(problem.INITIAL))
  if problem.IsGoal(node.State):
    return node, 0
  Frontier = []
  Frontier.append(node)
  reached = set()
  reached.add(tuple(problem.INITIAL))
  nodesExpanded = 0
  while (Frontier):
    ### print([str(n) for n in Frontier])
    node = Frontier.pop(0)
    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      ### print (s, "IsGoal=", problem.IsGoal(s))
      if problem.IsGoal(s):
        return child, nodesExpanded
      if s not in reached:
        reached.add(s)
        Frontier.append(child)
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Best-First Search

In [33]:
def BestFirstSearch(problem, f):
  node = Node(tuple(problem.INITIAL))
  Frontier = []
  heapq.heappush(Frontier,(f(node), node))
  reached = {}
  reached[tuple(problem.INITIAL)]=node
  nodesExpanded = 0
  while (Frontier):
    ##print([(x, str(n)) for (x,n) in Frontier])
    fValue, node = heapq.heappop(Frontier)
    ##print (node.State, "IsGoal=", problem.IsGoal(tuple(node.State)))
    if problem.IsGoal(tuple(node.State)):
      return node, nodesExpanded    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      if s not in reached or child.PathCost < reached[s].PathCost:
        reached[s] = child
        heapq.heappush(Frontier, (f(child), child))
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

# **Define the Puzzle-Solver for modularity**

In [71]:

def solve_puzzle(dimension, initial_state=None, search_algorithm=BreadthFirstSearch, heuristic=None):
    InitialState, Goal = configure_puzzle(dimension,initial_state)
   # Define or handle heuristic functions here
    UniformCostF = lambda n: n.PathCost
    AStarF = lambda n: n.PathCost + ManhattanDistance(n.State, Goal)  # Using Goal
    AStarFb = lambda n: n.PathCost + OutOfPlace(n.State, list(Goal))

    # Decision tree for heuristic selection
    if heuristic is not None:
        if heuristic == "Manhattan":
            heuristic_function = AStarF
        elif heuristic == "OutOfPlace":
            heuristic_function = AStarFb
        elif heuristic == "UniformCost":
            heuristic_function = UniformCostF
        else:
            raise ValueError("Invalid heuristic provided. Choose 'Manhattan', 'OutOfPlace', or 'UniformCost'.")
    else:
        # If heuristic is not provided and search_algorithm is BestFirstSearch, default to UniformCostF
        if search_algorithm is BestFirstSearch:
            heuristic_function = UniformCostF
        else:
            heuristic_function = None  # For BFS, no heuristic is used

    TileSliding = Problem()
    TileSliding.INITIAL = InitialState
    TileSliding.IsGoal = lambda s: s == Goal
    TileSliding.Actions = Actions
    TileSliding.Result = Result
    TileSliding.ActionCost = lambda s, a, sPrime: 1

    ret, cost = search_algorithm(TileSliding, heuristic_function)  # Use heuristic_function here
       # Check if ret is None (no solution found)
    if ret is None:
        print("No solution found within the search limits.")
        return None  # Indicate no solution
    sol = Solution(ret)
    print("Solution:", sol)
    print("Initial State:", TileSliding.INITIAL)
    print("Apply Moves:", ApplyMoves(sol, TileSliding.INITIAL))
    print("Length of Solution:", len(sol))
    print("Nodes Expanded:", cost)

## Problem 1 (Original no need to run this just collapse it)

In [21]:
Goal=tuple(GoalState)
TileSliding = Problem()
TileSliding.INITIAL = InitialState
TileSliding.IsGoal = lambda s: s==Goal
TileSliding.Actions = Actions
TileSliding.Result=Result
TileSliding.ActionCost = lambda s, a, sPrime: 1
print( TileSliding.IsGoal((GoalState)) )
print( Node(InitialState) )
print(1+TileSliding.ActionCost(1,2,3))

False
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 13, 14, 15], <none>
2


In [23]:
TileSliding.INITIAL = [1,2,3,4,5,6,0,7,8]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0], r 6


In [24]:
def Solution(node):
  if node.Parent==None:
    return []
  return Solution(node.Parent) + [node.Action]


In [25]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

['r', 'r', 'r']
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 13, 14, 15]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]


In [ ]:
TileSliding.INITIAL = [1,2,3,4,0,6,7,5,8]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

In [ ]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

In [ ]:
UniformCostF = lambda n: n.PathCost
AStarF = lambda n: n.PathCost+ManhattanDistance(n.State, GoalState)
TileSliding.INITIAL = [1,2,3,4,0,6,7,5,8]
ret, cost = BestFirstSearch(TileSliding, UniformCostF)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Nodes Expanded=", cost)

# **Problem 1 (rewritten for Modularity)**

In [48]:
InitialState = [1,2,3,4,5,6,0,7,8]
GoalState = [1,2,3,4,5,6,7,8,0]

In [52]:
# 3x3 puzzle
dimension =3
solve_puzzle(dimension=dimension,initial_state=InitialState)

Solution: ['r', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 0, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Nodes Expanded: 2


In [53]:
# BFS
solve_puzzle(3, InitialState, BreadthFirstSearch)

Solution: ['r', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 0, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Nodes Expanded: 2


In [55]:
# Uniform Cost Search
solve_puzzle(3, InitialState, BestFirstSearch)

Solution: ['r', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 0, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Nodes Expanded: 5


# Problem 2 (original  no need to run this. just collapse it)

In [ ]:
state1, sol = RandomWalk(GoalState, 300)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))

[7, 4, 0]
[6, 3, 1]
[8, 5, 2]
['u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'l', 'd', 'r', 'u', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'd', 'l', 'u', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'l', 'u', 'r', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'd', 'l', 'u', 'r', 'u', 'r', 'd', 'l', 'd', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'u', 'r', 'd', 'd', 'l', 'l', 'u', 'r', 'r', 'u', 'l', 'd', 'l', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'l', 'd', 'r', 'r', 'u', 'l', 'u', 'l',

In [ ]:
TileSliding.INITIAL = state1
ret, cost = BreadthFirstSearch(TileSliding)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Length of solution: ", len(sol))
print ("Nodes Expanded=", cost)

[1, 2, 3, 4, 5, 6, 7, 8, 0], d
['d', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'u', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'd']
[7, 4, 0, 6, 3, 1, 8, 5, 2]
[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 32126


In [ ]:
UniformCostF = lambda n: n.PathCost
TileSliding.INITIAL = state1
ret, cost = BestFirstSearch(TileSliding, UniformCostF)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Length of solution: ", len(sol))
print ("Nodes Expanded=", cost)

[1, 2, 3, 4, 5, 6, 7, 8, 0], d
['d', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'u', 'r', 'd', 'l', 'l', 'u', 'u', 'r', 'd', 'r', 'd']
[7, 4, 0, 6, 3, 1, 8, 5, 2]
[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 43525


# **Problem 2 (rewritten for modularity)**

In [ ]:
dimension = 3
walks = 300
state1, sol = RandomWalk(GoalState, walks, dimension)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))

In [ ]:
solve_puzzle(dimension, state1)

In [ ]:
solve_puzzle(dimension, state1, BreadthFirstSearch)

In [ ]:
solve_puzzle(dimension, state1, BestFirstSearch, UniformCostF)

In [ ]:
solve_puzzle(dimension, state1, BestFirstSearch, AStarF)

# Problem List

In [51]:
# findNum = 10
# randomWalkDistance = 300
# problemList = []
# for i in range(10):
#   state1, sol = RandomWalk(GoalState, 300)
#   problemList.append(state1)
# print (problemList)
num_problems_per_dimension = 15  # Total number of problems per dimension
random_walk_counts_3x3 = [5, 10, 20, 40, 80]  # For 3x3 puzzles
random_walk_counts_4x4 = [5, 10, 20, 40, 80]  # For 4x4 puzzles
num_problems_per_walk_count = 3  # Number of problems to generate for each walk count

problems_3x3 = []  # Problem list for 3x3 puzzles
problems_4x4 = []  # Problem list for 4x4 puzzles

# Generate problems for 3x3 puzzles
initial_state_3x3, goal_state_3x3 = configure_puzzle(3)
for walk_count in random_walk_counts_3x3:
    for _ in range(num_problems_per_walk_count):
        state, _ = RandomWalk(goal_state_3x3, walk_count, 3)  # Generate state using random walk
        problems_3x3.append(state)

# Generate problems for 4x4 puzzles
initial_state_4x4, goal_state_4x4 = configure_puzzle(4)
for walk_count in random_walk_counts_4x4:
    for _ in range(num_problems_per_walk_count):
        state, _ = RandomWalk(goal_state_4x4, walk_count, 4)  # Generate state using random walk
        problems_4x4.append(state)

print("Problem List for 3x3 Puzzles:", problems_3x3)
print("Problem List for 4x4 Puzzles:", problems_4x4)

Problem List for 3x3 Puzzles: [[1, 2, 3, 7, 4, 5, 8, 0, 6], [1, 5, 2, 0, 4, 3, 7, 8, 6], [2, 0, 3, 1, 5, 6, 4, 7, 8], [4, 1, 2, 6, 3, 8, 7, 5, 0], [4, 1, 2, 7, 0, 5, 8, 6, 3], [0, 4, 3, 2, 8, 5, 1, 7, 6], [2, 3, 7, 1, 6, 4, 5, 8, 0], [6, 2, 0, 1, 7, 4, 5, 8, 3], [2, 4, 0, 3, 1, 8, 7, 6, 5], [5, 3, 2, 1, 7, 4, 8, 6, 0], [1, 3, 2, 6, 4, 7, 0, 5, 8], [8, 3, 6, 5, 0, 2, 1, 7, 4], [5, 4, 0, 6, 1, 7, 8, 2, 3], [1, 4, 7, 8, 0, 5, 6, 2, 3], [0, 7, 4, 3, 8, 5, 6, 2, 1]]
Problem List for 4x4 Puzzles: [[1, 0, 2, 4, 5, 6, 3, 7, 9, 10, 11, 8, 13, 14, 15, 12], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 0, 13, 14, 12, 11], [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 11, 0, 14, 15, 12], [1, 2, 3, 4, 5, 6, 8, 12, 0, 9, 10, 15, 13, 14, 7, 11], [5, 2, 0, 3, 6, 1, 7, 4, 9, 10, 11, 8, 13, 14, 15, 12], [1, 2, 3, 4, 5, 6, 7, 0, 10, 11, 14, 8, 9, 13, 15, 12], [5, 1, 3, 4, 2, 10, 6, 8, 9, 7, 12, 15, 13, 14, 11, 0], [0, 6, 4, 8, 2, 7, 3, 12, 1, 9, 10, 15, 5, 13, 14, 11], [2, 6, 3, 4, 1, 10, 8, 12, 5, 15, 0, 11, 9, 13, 14, 7],

In [73]:
print(problems_3x3[7])

[6, 2, 0, 1, 7, 4, 5, 8, 3]


# **Bread First Search (Out of Place & Manhattan)**

In [63]:
dimension = 3


In [67]:
heuristic = "OutOfPlace"
for problem in problems_3x3:
  sol = solve_puzzle(dimension, problem, BreadthFirstSearch,heuristic)

Solution: ['l', 'u', 'r', 'r', 'd']
Initial State: [1, 2, 3, 7, 4, 5, 8, 0, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['r', 'u', 'r', 'd', 'd']
Initial State: [1, 5, 2, 0, 4, 3, 7, 8, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['l', 'd', 'd', 'r', 'r']
Initial State: [2, 0, 3, 1, 5, 6, 4, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 25
Solution: ['u', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Initial State: [4, 1, 2, 6, 3, 8, 7, 5, 0]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 336
Solution: ['r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'd']
Initial State: [4, 1, 2, 7, 0, 5, 8, 6, 3]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 531
Solution: ['d', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'r', 'd']
Initial State: [0, 4, 3, 2, 8, 5, 1, 7, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8,

In [68]:
heuristic = "Manhattan"
for problem in problems_3x3:
  sol = solve_puzzle(dimension, problem, BreadthFirstSearch,heuristic)

Solution: ['l', 'u', 'r', 'r', 'd']
Initial State: [1, 2, 3, 7, 4, 5, 8, 0, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['r', 'u', 'r', 'd', 'd']
Initial State: [1, 5, 2, 0, 4, 3, 7, 8, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['l', 'd', 'd', 'r', 'r']
Initial State: [2, 0, 3, 1, 5, 6, 4, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 25
Solution: ['u', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Initial State: [4, 1, 2, 6, 3, 8, 7, 5, 0]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 336
Solution: ['r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'd']
Initial State: [4, 1, 2, 7, 0, 5, 8, 6, 3]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 531
Solution: ['d', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'r', 'd']
Initial State: [0, 4, 3, 2, 8, 5, 1, 7, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8,

In [69]:
dimension=4

In [81]:
heuristic = "OutOfPlace"
for problem in problems_4x4:
  sol = solve_puzzle(dimension, problem, BreadthFirstSearch,heuristic)

Solution: ['r', 'd', 'r', 'd', 'd']
Initial State: [1, 0, 2, 4, 5, 6, 3, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 53
Solution: ['d', 'l', 'u', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 0, 13, 14, 12, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 36
Solution: ['u', 'r', 'r', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 11, 0, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['r', 'r', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 8, 12, 0, 9, 10, 15, 13, 14, 7, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 10
Nodes Expanded: 2248
Solution: ['l', 'd', 'l', 'u', 'r', 'r', 'r', 'd', 'd', 'd']
Initial State: [5, 2, 0, 3, 6, 1, 7, 4, 9, 10, 11, 

^^Runtime was not 2 mins. IDK why it says that. Took something like 10 mins to run this.

In [82]:
heuristic = "Manhattan"
for problem in problems_4x4:
  sol = solve_puzzle(dimension, problem, BreadthFirstSearch,heuristic)

Solution: ['r', 'd', 'r', 'd', 'd']
Initial State: [1, 0, 2, 4, 5, 6, 3, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 53
Solution: ['d', 'l', 'u', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 0, 13, 14, 12, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 36
Solution: ['u', 'r', 'r', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 11, 0, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 28
Solution: ['r', 'r', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 8, 12, 0, 9, 10, 15, 13, 14, 7, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 10
Nodes Expanded: 2248
Solution: ['l', 'd', 'l', 'u', 'r', 'r', 'r', 'd', 'd', 'd']
Initial State: [5, 2, 0, 3, 6, 1, 7, 4, 9, 10, 11, 

# **A* (Out of Place & Manhattan)**

In [75]:
dimension=3

In [76]:
heuristic = "OutOfPlace"
for problem in problems_3x3:
  sol = solve_puzzle(dimension, problem, BestFirstSearch, heuristic)

Solution: ['l', 'u', 'r', 'r', 'd']
Initial State: [1, 2, 3, 7, 4, 5, 8, 0, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['r', 'u', 'r', 'd', 'd']
Initial State: [1, 5, 2, 0, 4, 3, 7, 8, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['l', 'd', 'd', 'r', 'r']
Initial State: [2, 0, 3, 1, 5, 6, 4, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['u', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Initial State: [4, 1, 2, 6, 3, 8, 7, 5, 0]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 28
Solution: ['r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'd']
Initial State: [4, 1, 2, 7, 0, 5, 8, 6, 3]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 37
Solution: ['d', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'r', 'd']
Initial State: [0, 4, 3, 2, 8, 5, 1, 7, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
L

In [77]:
heuristic = "Manhattan"
for problem in problems_3x3:
  sol = solve_puzzle(dimension, problem, BestFirstSearch, heuristic)

Solution: ['l', 'u', 'r', 'r', 'd']
Initial State: [1, 2, 3, 7, 4, 5, 8, 0, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['r', 'u', 'r', 'd', 'd']
Initial State: [1, 5, 2, 0, 4, 3, 7, 8, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['l', 'd', 'd', 'r', 'r']
Initial State: [2, 0, 3, 1, 5, 6, 4, 7, 8]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['u', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Initial State: [4, 1, 2, 6, 3, 8, 7, 5, 0]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 12
Solution: ['r', 'd', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'd']
Initial State: [4, 1, 2, 7, 0, 5, 8, 6, 3]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of Solution: 10
Nodes Expanded: 16
Solution: ['d', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'r', 'd']
Initial State: [0, 4, 3, 2, 8, 5, 1, 7, 6]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 0]
L

In [78]:
dimension=4

In [79]:
heuristic = "OutOfPlace"
for problem in problems_4x4:
  sol = solve_puzzle(dimension, problem, BestFirstSearch, heuristic)

Solution: ['r', 'd', 'r', 'd', 'd']
Initial State: [1, 0, 2, 4, 5, 6, 3, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['d', 'l', 'u', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 0, 13, 14, 12, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 8
Solution: ['u', 'r', 'r', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 11, 0, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['r', 'r', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 8, 12, 0, 9, 10, 15, 13, 14, 7, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 10
Nodes Expanded: 36
Solution: ['l', 'd', 'l', 'u', 'r', 'r', 'r', 'd', 'd', 'd']
Initial State: [5, 2, 0, 3, 6, 1, 7, 4, 9, 10, 11, 8, 13

In [80]:
heuristic = "Manhattan"
for problem in problems_4x4:
  sol = solve_puzzle(dimension, problem, BestFirstSearch, heuristic)

Solution: ['r', 'd', 'r', 'd', 'd']
Initial State: [1, 0, 2, 4, 5, 6, 3, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['d', 'l', 'u', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 0, 13, 14, 12, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['u', 'r', 'r', 'r', 'd']
Initial State: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 11, 0, 14, 15, 12]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 5
Nodes Expanded: 5
Solution: ['r', 'r', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r']
Initial State: [1, 2, 3, 4, 5, 6, 8, 12, 0, 9, 10, 15, 13, 14, 7, 11]
Apply Moves: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of Solution: 10
Nodes Expanded: 14
Solution: ['l', 'd', 'l', 'u', 'r', 'r', 'r', 'd', 'd', 'd']
Initial State: [5, 2, 0, 3, 6, 1, 7, 4, 9, 10, 11, 8, 13

Based on the number of nodes expanded for some of these solutions it's not hard to see how this can be crucial for exploring a solution space in AI applications when you're dealing with a large number of parameters(see: N>1 000 000 000)